In [3]:
# Part 2, Step 5: Train the fair models

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score, f1_score
from aif360.datasets import BinaryLabelDataset
from aif360.algorithms.inprocessing import MetaFairClassifier
from aif360.metrics import ClassificationMetric

feature_cols = ['Age_bucket_u', 'Credit amount_bucket_u', 'Duration_bucket_u']
target_col = 'Risk'
protected_col = 'Sex'
df_fair = pd.read_csv('datasets/german_credit_data_ebiased.csv', delimiter=',')

# MetaFairClassifier assumes favorable predictions are encoded as 1.
# Keep a separate evaluation encoding so the final fairness metrics stay
# directly comparable to germancredit-validation.ipynb.
eval_label_map = {'good': 0, 'bad': 1}
meta_label_map = {'good': 1, 'bad': 0}
meta_label_inv_map = {1: 'good', 0: 'bad'}
sex_map = {'male': 1, 'female': 0}

df_fair['Risk_num_eval'] = df_fair[target_col].map(eval_label_map)
df_fair['Risk_num_meta'] = df_fair[target_col].map(meta_label_map)
df_fair['Sex_num'] = df_fair[protected_col].map(sex_map)

if df_fair['Sex_num'].isna().any():
    raise ValueError(f"Unmapped values found in {protected_col}: {df_fair.loc[df_fair['Sex_num'].isna(), protected_col].unique()}")

X = df_fair[feature_cols + ['Sex_num']]
y = df_fair[target_col]

X_train_fair, X_test_fair, y_train_fair, y_test_fair = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_train_encoded = encoder.fit_transform(X_train_fair[feature_cols])
X_test_encoded = encoder.transform(X_test_fair[feature_cols])
encoded_cols = encoder.get_feature_names_out(feature_cols)

train_encoded_df = pd.DataFrame(X_train_encoded, columns=encoded_cols, index=X_train_fair.index)
test_encoded_df = pd.DataFrame(X_test_encoded, columns=encoded_cols, index=X_test_fair.index)

train_bld_df = train_encoded_df.copy()
train_bld_df['Sex_num'] = X_train_fair['Sex_num'].values
train_bld_df['Risk_num'] = y_train_fair.map(meta_label_map).values

test_bld_df = test_encoded_df.copy()
test_bld_df['Sex_num'] = X_test_fair['Sex_num'].values
test_bld_df['Risk_num'] = y_test_fair.map(meta_label_map).values

dataset_train_fair = BinaryLabelDataset(
    favorable_label=1,
    unfavorable_label=0,
    df=train_bld_df,
    label_names=['Risk_num'],
    protected_attribute_names=['Sex_num']
)

dataset_test_fair = BinaryLabelDataset(
    favorable_label=1,
    unfavorable_label=0,
    df=test_bld_df,
    label_names=['Risk_num'],
    protected_attribute_names=['Sex_num']
)

fair_fdr_clf = MetaFairClassifier(tau=0, sensitive_attr='Sex_num', type='fdr')
fair_fdr_clf.fit(dataset_train_fair)
dataset_pred_fdr_meta = fair_fdr_clf.predict(dataset_test_fair)
y_pred_fdr = np.vectorize(meta_label_inv_map.get)(dataset_pred_fdr_meta.labels.ravel().astype(int))

fair_sr_clf = MetaFairClassifier(tau=0, sensitive_attr='Sex_num', type='sr')
fair_sr_clf.fit(dataset_train_fair)
dataset_pred_sr_meta = fair_sr_clf.predict(dataset_test_fair)

y_pred_sr = np.vectorize(meta_label_inv_map.get)(dataset_pred_sr_meta.labels.ravel().astype(int))

test_eval_df = pd.DataFrame({
    'Sex_num': X_test_fair['Sex_num'].values,
    'Risk_num': y_test_fair.map(eval_label_map).values
})
dataset_test_eval = BinaryLabelDataset(
    favorable_label=0,
    unfavorable_label=1,
    df=test_eval_df,
    label_names=['Risk_num'],
    protected_attribute_names=['Sex_num']
)

dataset_pred_fdr_eval = dataset_test_eval.copy()
dataset_pred_fdr_eval.labels = pd.Series(y_pred_fdr, index=X_test_fair.index).map(eval_label_map).values.reshape(-1, 1)

dataset_pred_sr_eval = dataset_test_eval.copy()
dataset_pred_sr_eval.labels = pd.Series(y_pred_sr, index=X_test_fair.index).map(eval_label_map).values.reshape(-1, 1)

privileged_groups = [{'Sex_num': 1}]
unprivileged_groups = [{'Sex_num': 0}]

def evaluate_model(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, pos_label='bad')
    print(f"{name}")
    print(f"  Accuracy: {acc:.4f}")
    print(f"  F1-score: {f1:.4f}")
    print("-" * 40)

def print_fairness_metrics(model_name, dataset_true, dataset_pred):
    metric = ClassificationMetric(
        dataset_true,
        dataset_pred,
        unprivileged_groups=unprivileged_groups,
        privileged_groups=privileged_groups
    )

    print(f"Statistical Parity Difference (SPD): {metric.statistical_parity_difference():.4f}")
    print(f"Disparate Impact (DI):              {metric.disparate_impact():.4f}")
    print(f"Equal Opportunity Difference:       {metric.equal_opportunity_difference():.4f}")
    print(f"Average Odds Difference:            {metric.average_odds_difference():.4f}")
    print(f"Theil Index:                        {metric.theil_index():.4f}")
    print("-" * 40)

evaluate_model('MetaFairClassifier (FDR)', y_test_fair, y_pred_fdr)
print_fairness_metrics('MetaFairClassifier (FDR)', dataset_test_eval, dataset_pred_fdr_eval)
evaluate_model('MetaFairClassifier (SR)', y_test_fair, y_pred_sr)
print_fairness_metrics('MetaFairClassifier (SR)', dataset_test_eval, dataset_pred_sr_eval)


MetaFairClassifier (FDR)
  Accuracy: 0.7000
  F1-score: 0.2105
----------------------------------------
Statistical Parity Difference (SPD): -0.0603
Disparate Impact (DI):              0.9357
Equal Opportunity Difference:       -0.0402
Average Odds Difference:            -0.0621
Theil Index:                        0.0966
----------------------------------------
MetaFairClassifier (SR)
  Accuracy: 0.7000
  F1-score: 0.2105
----------------------------------------
Statistical Parity Difference (SPD): -0.0603
Disparate Impact (DI):              0.9357
Equal Opportunity Difference:       -0.0402
Average Odds Difference:            -0.0621
Theil Index:                        0.0966
----------------------------------------


In [8]:
# Part 2, Step 5 (continued): ExponentiatedGradient – fair logistic regression
# Agarwal et al. (2018). Uses fairlearn directly (bypassing AIF360's broken
# wrapper which has a version-incompatibility bug with recent fairlearn).
#
# Reuses: X_train_fair, X_test_fair, y_train_fair, y_test_fair,
#         train_encoded_df, test_encoded_df, encoded_cols,
#         dataset_test_eval, eval_label_map, meta_label_map, meta_label_inv_map,
#         evaluate_model(), print_fairness_metrics()

import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'fairlearn', '-q'])

from sklearn.linear_model import LogisticRegression
from fairlearn.reductions import ExponentiatedGradient, DemographicParity, EqualizedOdds

# Build feature matrices with sensitive attribute included
X_train_egr = train_encoded_df.copy()
X_train_egr['Sex_num'] = X_train_fair['Sex_num'].values

X_test_egr = test_encoded_df.copy()
X_test_egr['Sex_num'] = X_test_fair['Sex_num'].values

sensitive_train = X_train_fair['Sex_num'].values
sensitive_test  = X_test_fair['Sex_num'].values

y_train_num = y_train_fair.map(meta_label_map)  # 1=good, 0=bad (favorable=1)

lr_estimator = LogisticRegression(solver='lbfgs', max_iter=1000, random_state=42)

for constraint_name, constraint in [
    ('DemographicParity', DemographicParity()),
    ('EqualizedOdds',     EqualizedOdds()),
]:
    egr = ExponentiatedGradient(estimator=lr_estimator, constraints=constraint)
    egr.fit(X_train_egr, y_train_num, sensitive_features=sensitive_train)

    y_pred_num = egr.predict(X_test_egr)  # 1=good, 0=bad
    y_pred_egr = np.vectorize(meta_label_inv_map.get)(y_pred_num.astype(int))

    dataset_pred_egr_eval = dataset_test_eval.copy()
    dataset_pred_egr_eval.labels = (
        pd.Series(y_pred_egr, index=X_test_fair.index)
        .map(eval_label_map)
        .values
        .reshape(-1, 1)
    )

    evaluate_model(f'ExponentiatedGradient ({constraint_name})', y_test_fair, y_pred_egr)
    print_fairness_metrics(f'ExponentiatedGradient ({constraint_name})', dataset_test_eval, dataset_pred_egr_eval)



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip


ExponentiatedGradient (DemographicParity)
  Accuracy: 0.7033
  F1-score: 0.2261
----------------------------------------
Statistical Parity Difference (SPD): -0.0079
Disparate Impact (DI):              0.9914
Equal Opportunity Difference:       0.0075
Average Odds Difference:            -0.0054
Theil Index:                        0.0963
----------------------------------------
ExponentiatedGradient (EqualizedOdds)
  Accuracy: 0.7000
  F1-score: 0.2373
----------------------------------------
Statistical Parity Difference (SPD): -0.0095
Disparate Impact (DI):              0.9895
Equal Opportunity Difference:       -0.0270
Average Odds Difference:            0.0102
Theil Index:                        0.1032
----------------------------------------
